# 01 — Tox21 Exploratory Data Analysis

탐색적 데이터 분석: 멀티레이블 분포·상관, 결측치, 물리화학 특성(LogP/MW), 대표 분자 구조.

**실행 순서**: 위에서부터 셀을 차례로, 또는 `런타임 > 모두 실행`.

In [ ]:
# === STEP 1: 환경 설정 (가장 먼저 이 셀 실행) ===========================
import os, sys, subprocess

REPO = "Tox21-Toxicity-Classifier"
REPO_URL = "https://github.com/zqvo04/Tox21-Toxicity-Classifier.git"
ROOT = "/content/" + REPO

# (1) 레포 준비: 없으면 clone, 이미 있으면 최신으로 git pull(오래된 캐시 방지)
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", REPO_URL, ROOT], check=False)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("작업 디렉토리:", os.getcwd())

# (2) 의존성 설치 (idempotent). 'rdkit'(공식) 사용 — 'rdkit-pypi'는 최신
#     Python에서 설치 불가. 데이터/특징화는 RDKit으로 직접 처리(견고).
pkgs = ["rdkit", "torch-geometric",
        "pandas", "scikit-learn", "matplotlib", "seaborn", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# (3) 코드 버전 검증: 최신 RDKit 기반 dataset.py 인지 확인 (deepchem 의존 없음)
import importlib
import src.dataset as _ds
importlib.reload(_ds)
if not hasattr(_ds, "load_tox21_dataframe"):
    raise RuntimeError(
        "⚠️ 오래된 dataset.py 입니다 (deepchem 버전). 다음을 확인하세요:\n"
        "  1) 최신 코드를 GitHub 에 push 했는지\n"
        "  2) Colab 상단 [런타임 > 세션 다시 시작 및 모두 삭제] 후 이 셀을 다시 실행\n"
        "     (오래된 clone 폴더가 남아 있으면 갱신이 안 됩니다)")

# (4) import 점검. 실패 시 런타임 재시작 안내.
try:
    import rdkit, torch, torch_geometric, sklearn
    print("✅ rdkit", rdkit.__version__,
          "| torch", torch.__version__, "| pyg", torch_geometric.__version__,
          "| gpu", torch.cuda.is_available(), "| dataset.py: 최신(RDKit) ✔")
except Exception as e:
    print("⚠️ import 실패:", repr(e))
    print("➡️  상단 메뉴 [런타임 > 런타임 다시 시작] 후, 이 셀을 한 번 더 실행하세요.")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
from src.dataset import label_matrix
FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)

## 1. 데이터 로딩 & 레이블 행렬
(첫 실행 시 MoleculeNet Tox21 CSV 를 내려받아 캐시; RDKit 으로 파싱)

In [ ]:
y, tasks, smiles = label_matrix()
print('Molecules:', len(smiles), '| Tasks:', len(tasks))
df_y = pd.DataFrame(y, columns=tasks)
df_y.head()

## 2. 레이블 분포 & 결측치

In [ ]:
pos_rate = df_y.apply(lambda c: np.nanmean(c))
missing_rate = df_y.isna().mean()
stats = pd.DataFrame({'positive_rate': pos_rate, 'missing_rate': missing_rate})
display(stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
stats['positive_rate'].plot.bar(ax=axes[0], color='crimson')
axes[0].set_title('Positive (toxic) rate per task'); axes[0].set_ylabel('rate')
stats['missing_rate'].plot.bar(ax=axes[1], color='steelblue')
axes[1].set_title('Missing-label rate per task'); axes[1].set_ylabel('rate')
plt.tight_layout(); plt.savefig(f'{FIG}/label_stats.png', dpi=150); plt.show()

In [ ]:
corr = df_y.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': .8})
plt.title('Tox21 endpoint label correlation')
plt.tight_layout(); plt.savefig(f'{FIG}/label_correlation_heatmap.png', dpi=150); plt.show()

## 3. 물리화학적 특성 분포 (LogP / MW / TPSA)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

records = []
for smi in smiles:
    m = Chem.MolFromSmiles(smi)
    if m is None:
        continue
    records.append({'smiles': smi, 'MW': Descriptors.MolWt(m),
                    'LogP': Descriptors.MolLogP(m), 'TPSA': Descriptors.TPSA(m)})
props = pd.DataFrame(records)
props.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(props['MW'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Molecular Weight'); axes[0].axvline(500, ls='--', c='r')
sns.histplot(props['LogP'], bins=50, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('LogP'); axes[1].axvline(5, ls='--', c='r')
sns.histplot(props['TPSA'], bins=50, kde=True, ax=axes[2], color='purple')
axes[2].set_title('TPSA')
plt.tight_layout(); plt.savefig(f'{FIG}/physchem_distributions.png', dpi=150); plt.show()

## 4. 대표 독성 분자 구조 (SR-MMP 양성 샘플)

In [ ]:
task = 'SR-MMP'; ti = tasks.index(task)
toxic = [s for s, lab in zip(smiles, y[:, ti]) if lab == 1][:12]
mols = [Chem.MolFromSmiles(s) for s in toxic]; mols = [m for m in mols if m]
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 180),
                           legends=[f'{task}=1']*len(mols))
def save_grid(img, path):  # robust across PIL / IPython image return types
    try: img.save(path)
    except AttributeError:
        open(path, 'wb').write(img.data)
save_grid(img, f'{FIG}/example_toxic_molecules.png'); img

## 요약
- 12개 독성 엔드포인트, 클래스 불균형이 큼 (positive rate 낮음)
- 결측 레이블 비율이 task마다 다름 → **마스킹 필수**
- 대부분 분자가 Lipinski 규칙(MW<500, LogP<5) 범위